# Lab 6
- **Student Name**: Daniel Alejandro Castillo Martin
- **Student ID**: N10019284
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
---

## Overview

This lab investigates parameter-efficient fine-tuning (PEFT) by adapting the pretrained **HuggingFaceTB/SmolLM2-135M** language model to a text classification task using Low-Rank Adaptation (LoRA).

The experiments compare full fine-tuning against LoRA at different ranks, evaluating classification accuracy, trainable parameter counts, computational cost, and memory consumption.

## Objectives

1. Evaluate the pre-trained model using a zero-shot classification baseline.
2. Implement answer-only loss masking to focus training on classification labels.
3. Implement LoRA from scratch and validate its behavior against the PEFT library.
4. Compare full fine-tuning with LoRA ranks 1, 4, and 16 using controlled experiments.
5. Predict and measure memory consumption across fine-tuning configurations.
6. Analyze the trade-offs between accuracy, parameter efficiency, and computational resources.


## 1. Imports and Setup




- This section initializes the libraries, configuration parameters, and execution environment required for the experiments.

- The implementation uses PyTorch for tensor operations and model training, Hugging Face Transformers for pretrained language models, and PEFT for validating the custom LoRA implementation.

- The provided `lab06_starter.py` module supplies standardized dataset loading, training, evaluation, and memory-measurement utilities, ensuring consistent experimental conditions.

### 1.1 Required Libraries

In [12]:
# Import general libraries for reproducibility and paths
import sys
import subprocess

# Import libraries for data analysis and visualization
import matplotlib.pyplot as plt
import pandas as pd

# Import PyTorch for model operations and verification
import torch
import torch.nn as nn
import torch.nn.functional as F

# Import the provided lab starter
from scripts import lab06_starter as S

# Import project custom modules
from src.config import (MODEL_NAME, TASK_NAME, SEEDS, DATA_SEED, N_TRAIN, N_TEST, LORA_RANKS, TRAINING_STEPS, BATCH_SIZE, FULL_LR, LORA_LR, OUTPUT_DIR, FIGURES_DIR, create_output_directories)
from src.utils import set_seed, get_hardware_info
from src.losses import answer_only_labels

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

### 1.2 Experiment Configuration


- The experiment settings are defined centrally to ensure consistency and reproducibility across fine-tuning methods.

- All configurations will use the same dataset, evaluation split, training steps, batch size, and random seeds. Learning rates may differ between full fine-tuning and LoRA, as permitted by the assignment.

- Generated results and figures will be stored separately from the source code.

In [2]:
# Initialize the experiment output directories.
create_output_directories()

# Display the experimental configuration.
print(f'Model: {MODEL_NAME}')
print(f'Task: {TASK_NAME}')
print(f'Training examples: {N_TRAIN}')
print(f'Test examples: {N_TEST}')
print(f'Data seed: {DATA_SEED}')
print(f'Training seeds: {SEEDS}')
print(f'LoRA ranks: {LORA_RANKS}')
print(f'Training steps: {TRAINING_STEPS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Full fine-tuning LR: {FULL_LR}')
print(f'LoRA LR: {LORA_LR}')

Model: HuggingFaceTB/SmolLM2-135M
Task: emotion
Training examples: 600
Test examples: 500
Data seed: 0
Training seeds: [0, 1, 2]
LoRA ranks: [1, 4, 16]
Training steps: 120
Batch size: 8
Full fine-tuning LR: 5e-05
LoRA LR: 0.001


### 1.3 Reproducibility and Hardware

- Reproducibility is important for ensuring that observed differences between full fine-tuning and LoRA configurations are attributable to the methods being compared rather than uncontrolled randomness.

- The experiments use fixed random seeds for Python and PyTorch, consistent dataset sampling, and identical training budgets across configurations. Each experiment will be repeated using seeds 0, 1, and 2.

- The available hardware and software environment are also recorded because training time and GPU memory consumption depend on the execution platform.

In [3]:
# Initialize reproducibility with the first experimental seed
set_seed(SEEDS[0])

# Retrieve and display the execution environment
hardware_info = get_hardware_info()

for key, value in hardware_info.items():
    print(f'{key}: {value}')

Python: 3.12.10
PyTorch: 2.8.0+cu128
CUDA available: True
CUDA version: 12.8
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


### 1.4 Starter Script Validation

- Before implementing the custom LoRA components, the provided `lab06_starter.py` script is executed without modification to validate the experimental environment.

- The starter loads the pretrained SmolLM2-135M model and the emotion classification dataset, evaluates zero-shot accuracy, and performs a short 30-step LoRA training run using PEFT.

- This preliminary test verifies that model loading, dataset preparation, evaluation, and GPU-accelerated training work correctly before conducting the controlled experiments.

In [ ]:
# Configure execution to test that the starter script works
result = subprocess.run([sys.executable, 'scripts/lab06_starter.py'], capture_output = True, text = True, encoding = 'utf-8',errors = 'replace')

# Display the execution output
print(result.stdout)

# Validate successful execution
assert result.returncode == 0, 'Starter script execution failed.'
print('Starter script validation completed successfully.')

device: cuda
example prompt: 'Text: i may feel that way but the fact that stories created by adults that are meant for children contain messages that are not so innocent really makes me wonder who exactly is\nEmotion:' answer: ' joy'
zero-shot accuracy (chance 0.167): 0.35
LoRA r=8, 30 steps: accuracy 0.510, trainable 460,800 of 134,975,808, 0.12 s/step

Starter script validation completed successfully.


#### Validation Results

The original starter script completed successfully using CUDA acceleration on the NVIDIA GeForce RTX 3060 Laptop GPU.

- **Zero-shot accuracy:** 35.0%, exceeding the random-chance baseline of 16.7%.
- **LoRA accuracy:** 51.0% after 30 training steps, representing a 16-percentage-point improvement.
- **Trainable parameters:** 460,800 out of 134,975,808 total parameters (approximately 0.34%).
- **Training time:** Approximately 0.12 seconds per step.

The results confirm that pretrained model loading, dataset preparation, evaluation, and LoRA fine-tuning are operational.

Although the preliminary LoRA run improved classification accuracy, this result alone does not establish whether LoRA matches full fine-tuning or how adaptation rank affects performance. Those comparisons require controlled experiments using identical datasets, training budgets, and random seeds.

## 2. Zero-Shot Baseline and Answer-Only Loss

- This section evaluates the pretrained language model before fine-tuning and introduces a task-specific loss function for emotion classification.

- The model performs classification without an additional classification head. Instead, each input is formatted as a text prompt ending with an emotion cue, and the model predicts the most likely label token.

- The zero-shot baseline provides a reference for measuring the effectiveness of subsequent fine-tuning experiments.

### 2.1 Model and Dataset Preparation

- The experiment uses **HuggingFaceTB/SmolLM2-135M**, a pretrained decoder-only language model, and the **dair-ai/emotion** dataset.

- The task includes six emotion categories: sadness, joy, love, anger, fear, and surprise.

- For controlled evaluation, 600 training examples and 500 test examples are selected using a fixed data seed. The same examples will be reused across all fine-tuning configurations.

In [10]:
# Load the pretrained language model and tokenizer
model, tokenizer = S.load_model()

# Load the emotion classification dataset using a fixed sampling seed
train_examples, test_examples, label_ids = S.load_task(
    task = TASK_NAME,
    n_train = N_TRAIN,
    n_test = N_TEST,
    data_seed = DATA_SEED,
    tokenizer = tokenizer
)

# Display the dataset and classification configuration
print(f'\nModel: {MODEL_NAME}')
print(f'Device: {S.DEVICE}')
print(f'Training examples: {len(train_examples)}')
print(f'Test examples: {len(test_examples)}')
print(f'Emotion classes: {S.TASKS[TASK_NAME]['words']}')


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1361.31it/s]



Model: HuggingFaceTB/SmolLM2-135M
Device: cuda
Training examples: 600
Test examples: 500
Emotion classes: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [11]:
# Inspect the first tokenized classification example
example = train_examples[0]

print('\nExample prompt:')
print(tokenizer.decode(example['prompt_ids']))

print('\nExpected label:')
print(S.TASKS[TASK_NAME]['words'][example['label']])

print('\nAnswer token:')
print(repr(tokenizer.decode([example['answer_id']])))


Example prompt:
Text: i may feel that way but the fact that stories created by adults that are meant for children contain messages that are not so innocent really makes me wonder who exactly is
Emotion:

Expected label:
joy

Answer token:
' joy'


### 2.2 Zero-Shot Baseline Evaluation


- Zero-shot evaluation measures the pretrained model's ability to perform emotion classification without task-specific fine-tuning.

- The model receives a text prompt ending with `Emotion:` and predicts the most likely emotion by comparing the logits associated with the six predefined label tokens.

- Since the task contains six classes, the uniform random-chance baseline is approximately 16.67%. The zero-shot accuracy provides a reference for measuring the improvements or regressions produced by subsequent fine-tuning experiments.

In [ ]:
# Evaluate the pretrained model without fine-tuning
zero_shot_accuracy = S.evaluate(
    model = model,
    tok = tokenizer,
    test = test_examples,
    label_ids = label_ids,
    batch_size = 32
    )

# Calculate the uniform random-chance baseline
chance_accuracy = 1 / len(label_ids)

# Measure the difference between zero-shot accuracy and chance.
baseline_improvement = zero_shot_accuracy - chance_accuracy

# Display baseline evaluation results.
print(f'Zero-shot accuracy: {zero_shot_accuracy:.2%}')
print(f'Chance accuracy: {chance_accuracy:.2%}')
print(f'Improvement over chance: {baseline_improvement:.2%}')

Zero-shot accuracy: 32.60%
Chance accuracy: 16.67%
Improvement over chance: 15.93%


#### Baseline Insights

- The pretrained SmolLM2-135M model achieved a **zero-shot accuracy of 32.60%** on 500 test examples, compared with a uniform random-chance baseline of **16.67%**.

- This represents an improvement of **15.93 percentage points** over chance, indicating that the model can associate input text with emotion labels without task-specific fine-tuning.

- However, the accuracy remains relatively low, with 337 of the 500 examples classified incorrectly. This suggests that pretrained language representations alone are insufficient for reliable performance on this classification task.

- The measured zero-shot accuracy will serve as the reference for evaluating whether full fine-tuning and LoRA improve classification performance under controlled experimental conditions.

- **Limitations:** The result reflects one pretrained model, one dataset, and one fixed test sample. Additionally, uniform chance is a theoretical reference rather than an estimate of the accuracy of a majority-class classifier.

### 2.3  Answer-Only Loss Implementation and Verification

- The starter's default `all_tokens_labels()` function computes next-token prediction loss across all non-padding tokens, including the input prompt and classification answer.

- To align training with the emotion classification objective, a custom `answer_only_labels()` function is implemented in `src/losses.py`. The function masks all prompt and padding positions using `-100`, allowing only the target answer token to contribute to the loss.

- The implementation is validated through assertions to ensure that exactly one token per example is scored, no prompt or padding tokens contribute to the loss, and the supervised token matches the expected emotion label.


In [31]:
# Prepare a batch of tokenized training examples
input_ids, attention_mask, prompt_lens = S.pad_batch(train_examples[:8], tokenizer.pad_token_id)

# Generate labels using the custom answer-only loss function
labels = answer_only_labels(input_ids, attention_mask, prompt_lens)

# Identify which tokens contribute to the loss
scored_positions = labels != -100

# Verify that exactly one token per example is scored
assert torch.all(scored_positions.sum(dim = 1) == 1)
print('Passed!')

Passed!


In [32]:
# Verify that the answer token is the scored position
batch_indices = torch.arange(input_ids.size(0))
assert torch.all(scored_positions[batch_indices, prompt_lens])
print(f'Batch size: {input_ids.size(0)}')
print('Passed!')

Batch size: 8
Passed!


In [33]:
# Verify that prompt tokens are excluded
token_positions = torch.arange(input_ids.size(1)).unsqueeze(0)
prompt_mask = token_positions < prompt_lens.unsqueeze(1)
assert torch.all(labels[prompt_mask] == -100)

# Verify that padding tokens are excluded.
assert torch.all(labels[attention_mask == 0] == -100)
print('Passed!')

Passed!


In [34]:
# Verify that scored tokens match the expected emotion labels
expected_answers = torch.tensor([example['answer_id'] for example in train_examples[:8]])
assert torch.equal(labels[batch_indices, prompt_lens], expected_answers)
print('Passed!')

Passed!


### 2.4 All-Token vs. Answer-Only Loss: Interpretation

- The starter's all-token loss trains the model to predict the next token throughout the input prompt, effectively optimizing general language modeling alongside the classification answer.

- However, predicting the prompt text is not the skill being evaluated in this experiment; answer-only loss focuses the training objective on predicting the correct emotion label.


#### Verification Insights

- The custom answer-only masking function was verified using eight tokenized training examples. All assertions confirmed that exactly one token per example contributed to the loss, with prompt and padding tokens excluded and the remaining supervised token matching the expected emotion label.

- These checks establish that the masking implementation follows the intended classification objective before it is used in the fine-tuning experiments.